<a href="https://colab.research.google.com/github/Kira-Stargazer/Aquadex-AI/blob/main/Aquadex_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [42]:
# ============================================================
# CELL 1 — GOOGLE DRIVE + ENVIRONMENT CHECK
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

import os
import sys

PROJECT_DIR = "/content/drive/MyDrive/Marine_Debris_Project"

print("Python:", sys.version)
print("Project directory:", PROJECT_DIR)
print("Project exists:", os.path.exists(PROJECT_DIR))

print("\nChecking installed packages...")

import numpy as np
import pandas as pd
import torch
import streamlit as st
import ultralytics

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Streamlit:", st.__version__)
print("Ultralytics:", ultralytics.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: Running on CPU")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Python: 3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]
Project directory: /content/drive/MyDrive/Marine_Debris_Project
Project exists: True

Checking installed packages...
NumPy: 2.1.3
Pandas: 2.2.3
Streamlit: 1.62.0
Ultralytics: 8.4.127
PyTorch: 2.11.0+cpu
CUDA available: False


In [43]:
# ============================================================
# CELL 2 — FIND YOLO MODEL
# ============================================================

import os

MODEL_CANDIDATES = [
    "/content/drive/MyDrive/Marine_Debris_Project/training/sss_shipwreck_v2_1024/weights/best.pt",
    "/content/drive/MyDrive/Marine_Debris_Project/training/sss_shipwreck_v2_1024/weights/last.pt",
    "/content/drive/MyDrive/Marine_Debris_Project/training/sss_shipwreck_v1/weights/best.pt",
    "/content/drive/MyDrive/Marine_Debris_Project/training/sss_shipwreck_v1/weights/last.pt",
]

MODEL_PATH = None

print("Searching for YOLO model files...\n")

for path in MODEL_CANDIDATES:
    if os.path.exists(path):
        print("FOUND:", path)

        # Prefer v2 best.pt
        if MODEL_PATH is None:
            MODEL_PATH = path

if MODEL_PATH is None:
    raise FileNotFoundError(
        "No YOLO model found. Check your Google Drive training folder."
    )

print("\nSelected model:")
print(MODEL_PATH)

Searching for YOLO model files...

FOUND: /content/drive/MyDrive/Marine_Debris_Project/training/sss_shipwreck_v2_1024/weights/best.pt
FOUND: /content/drive/MyDrive/Marine_Debris_Project/training/sss_shipwreck_v2_1024/weights/last.pt
FOUND: /content/drive/MyDrive/Marine_Debris_Project/training/sss_shipwreck_v1/weights/best.pt
FOUND: /content/drive/MyDrive/Marine_Debris_Project/training/sss_shipwreck_v1/weights/last.pt

Selected model:
/content/drive/MyDrive/Marine_Debris_Project/training/sss_shipwreck_v2_1024/weights/best.pt


In [44]:
# ============================================================
# CELL 3 — OUTPUT DIRECTORIES
# ============================================================

import os

PREDICTION_DIR = os.path.join(
    PROJECT_DIR,
    "predictions",
    "aquadex_ai"
)

IMAGE_DIR = os.path.join(
    PREDICTION_DIR,
    "images"
)

JSON_DIR = os.path.join(
    PREDICTION_DIR,
    "json"
)

CSV_DIR = os.path.join(
    PREDICTION_DIR,
    "csv"
)

os.makedirs(IMAGE_DIR, exist_ok=True)
os.makedirs(JSON_DIR, exist_ok=True)
os.makedirs(CSV_DIR, exist_ok=True)

print("Output directories created:\n")

print("Images:")
print(IMAGE_DIR)

print("\nJSON:")
print(JSON_DIR)

print("\nCSV:")
print(CSV_DIR)

Output directories created:

Images:
/content/drive/MyDrive/Marine_Debris_Project/predictions/aquadex_ai/images

JSON:
/content/drive/MyDrive/Marine_Debris_Project/predictions/aquadex_ai/json

CSV:
/content/drive/MyDrive/Marine_Debris_Project/predictions/aquadex_ai/csv


In [45]:
# ============================================================
# CELL 4 — CREATE COMPLETE AQUADEX AI STREAMLIT APP
# ============================================================

import os

APP_PATH = "/content/aquadex_app.py"

app_code = r'''
import os
import io
import json
import csv
from datetime import datetime

import streamlit as st
from PIL import Image, ImageDraw
from PIL.ExifTags import TAGS, GPSTAGS

from ultralytics import YOLO


# ============================================================
# CONFIGURATION
# ============================================================

PROJECT_DIR = "/content/drive/MyDrive/Marine_Debris_Project"

OUTPUT_DIR = os.path.join(
    PROJECT_DIR,
    "predictions",
    "aquadex_ai"
)

IMAGE_DIR = os.path.join(
    OUTPUT_DIR,
    "images"
)

JSON_DIR = os.path.join(
    OUTPUT_DIR,
    "json"
)

CSV_DIR = os.path.join(
    OUTPUT_DIR,
    "csv"
)

os.makedirs(IMAGE_DIR, exist_ok=True)
os.makedirs(JSON_DIR, exist_ok=True)
os.makedirs(CSV_DIR, exist_ok=True)


MODEL_CANDIDATES = [
    os.path.join(
        PROJECT_DIR,
        "training",
        "sss_shipwreck_v2_1024",
        "weights",
        "best.pt"
    ),

    os.path.join(
        PROJECT_DIR,
        "training",
        "sss_shipwreck_v2_1024",
        "weights",
        "last.pt"
    ),

    os.path.join(
        PROJECT_DIR,
        "training",
        "sss_shipwreck_v1",
        "weights",
        "best.pt"
    ),

    os.path.join(
        PROJECT_DIR,
        "training",
        "sss_shipwreck_v1",
        "weights",
        "last.pt"
    )
]


# ============================================================
# PAGE CONFIG
# ============================================================

st.set_page_config(
    page_title="Aquadex AI",
    page_icon="🌊",
    layout="wide",
    initial_sidebar_state="expanded"
)


# ============================================================
# CUSTOM CSS
# ============================================================

st.markdown(
    """
    <style>

    .main-title {
        font-size: 42px;
        font-weight: 800;
        margin-bottom: 5px;
    }

    .subtitle {
        font-size: 18px;
        opacity: 0.75;
        margin-bottom: 25px;
    }

    .status-box {
        padding: 18px;
        border-radius: 12px;
        text-align: center;
        font-size: 24px;
        font-weight: 800;
        margin-top: 20px;
        margin-bottom: 25px;
    }

    .detected {
        background-color: #0b4d27;
        color: white;
    }

    .not-detected {
        background-color: #5a1c1c;
        color: white;
    }

    .metric-card {
        padding: 20px;
        border-radius: 12px;
        border: 1px solid rgba(128,128,128,0.3);
        text-align: center;
    }

    .metric-number {
        font-size: 32px;
        font-weight: 800;
    }

    .metric-label {
        opacity: 0.7;
        font-size: 14px;
    }

    .gps-box {
        padding: 18px;
        border-radius: 12px;
        border: 1px solid rgba(128,128,128,0.3);
        margin-top: 10px;
        margin-bottom: 20px;
    }

    </style>
    """,
    unsafe_allow_html=True
)


# ============================================================
# GPS FUNCTIONS
# ============================================================

def _rational_to_float(value):
    """
    Convert EXIF rational values to float.
    Supports older PIL and newer PIL representations.
    """

    try:
        return float(value)
    except Exception:
        try:
            return float(value.numerator) / float(value.denominator)
        except Exception:
            try:
                return float(value[0]) / float(value[1])
            except Exception:
                raise ValueError("Unable to convert GPS value")


def gps_to_decimal(values):
    """
    Convert GPS degrees/minutes/seconds to decimal degrees.
    """

    degrees = _rational_to_float(values[0])
    minutes = _rational_to_float(values[1])
    seconds = _rational_to_float(values[2])

    return degrees + (minutes / 60.0) + (seconds / 3600.0)


def extract_gps_from_image(image):
    """
    Extract GPS coordinates from image EXIF metadata.

    Returns:
        latitude,
        longitude
    """

    try:

        exif = image.getexif()

        if not exif:
            return None, None

        gps_raw = None

        for key, value in exif.items():

            tag_name = TAGS.get(key, key)

            if tag_name == "GPSInfo":
                gps_raw = value
                break

        if gps_raw is None:
            return None, None

        gps_data = {}

        for key, value in gps_raw.items():
            gps_data[GPSTAGS.get(key, key)] = value

        if "GPSLatitude" not in gps_data:
            return None, None

        if "GPSLongitude" not in gps_data:
            return None, None

        latitude = gps_to_decimal(
            gps_data["GPSLatitude"]
        )

        longitude = gps_to_decimal(
            gps_data["GPSLongitude"]
        )

        latitude_ref = gps_data.get(
            "GPSLatitudeRef",
            "N"
        )

        longitude_ref = gps_data.get(
            "GPSLongitudeRef",
            "E"
        )

        if isinstance(latitude_ref, bytes):
            latitude_ref = latitude_ref.decode(errors="ignore")

        if isinstance(longitude_ref, bytes):
            longitude_ref = longitude_ref.decode(errors="ignore")

        if str(latitude_ref).upper() == "S":
            latitude = -latitude

        if str(longitude_ref).upper() == "W":
            longitude = -longitude

        return latitude, longitude

    except Exception as e:

        print("GPS extraction error:", e)

        return None, None


# ============================================================
# MODEL LOADING
# ============================================================

@st.cache_resource
def load_model():

    selected_model = None

    for candidate in MODEL_CANDIDATES:

        if os.path.exists(candidate):

            selected_model = candidate
            break

    if selected_model is None:

        raise FileNotFoundError(
            "No YOLO model was found in Google Drive."
        )

    model = YOLO(selected_model)

    return model, selected_model


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def safe_class_name(names, class_id):

    try:

        if isinstance(names, dict):
            return str(names.get(class_id, "shipwreck"))

        return str(names[class_id])

    except Exception:

        return "shipwreck"


def create_detection_rows(results, confidence_threshold):

    detections = []

    for result in results:

        boxes = result.boxes

        if boxes is None:
            continue

        for i in range(len(boxes)):

            confidence = float(
                boxes.conf[i].item()
            )

            if confidence < confidence_threshold:
                continue

            class_id = int(
                boxes.cls[i].item()
            )

            xyxy = boxes.xyxy[i].tolist()

            x1 = int(round(xyxy[0]))
            y1 = int(round(xyxy[1]))
            x2 = int(round(xyxy[2]))
            y2 = int(round(xyxy[3]))

            class_name = safe_class_name(
                result.names,
                class_id
            )

            detections.append(
                {
                    "detection_id": len(detections) + 1,
                    "class_id": class_id,
                    "class": class_name,
                    "confidence": round(
                        confidence,
                        4
                    ),
                    "bbox": {
                        "x1": x1,
                        "y1": y1,
                        "x2": x2,
                        "y2": y2
                    }
                }
            )

    return detections


def annotate_image(
    image,
    results,
    confidence_threshold
):

    annotated = image.copy()

    draw = ImageDraw.Draw(
        annotated,
        "RGBA"
    )

    detection_number = 0

    for result in results:

        boxes = result.boxes

        if boxes is None:
            continue

        for i in range(len(boxes)):

            confidence = float(
                boxes.conf[i].item()
            )

            if confidence < confidence_threshold:
                continue

            xyxy = boxes.xyxy[i].tolist()

            x1 = int(round(xyxy[0]))
            y1 = int(round(xyxy[1]))
            x2 = int(round(xyxy[2]))
            y2 = int(round(xyxy[3]))

            class_id = int(
                boxes.cls[i].item()
            )

            class_name = safe_class_name(
                result.names,
                class_id
            )

            detection_number += 1

            # Bounding box
            draw.rectangle(
                [x1, y1, x2, y2],
                outline=(30, 100, 255, 255),
                width=4
            )

            # Label
            label = (
                f"{class_name} "
                f"{confidence * 100:.1f}%"
            )

            try:
                bbox = draw.textbbox(
                    (x1, y1),
                    label
                )

                text_width = bbox[2] - bbox[0]
                text_height = bbox[3] - bbox[1]

            except Exception:

                text_width = len(label) * 8
                text_height = 18

            label_y = max(
                0,
                y1 - text_height - 8
            )

            draw.rectangle(
                [
                    x1,
                    label_y,
                    x1 + text_width + 10,
                    label_y + text_height + 6
                ],
                fill=(30, 80, 220, 220)
            )

            draw.text(
                (x1 + 5, label_y + 2),
                label,
                fill=(255, 255, 255, 255)
            )

    return annotated


def make_json_bytes(report):

    return json.dumps(
        report,
        indent=4,
        ensure_ascii=False
    ).encode("utf-8")


def make_csv_bytes(
    detections,
    filename,
    latitude,
    longitude,
    gps_source,
    timestamp
):

    output = io.StringIO()

    fieldnames = [
        "image",
        "timestamp",
        "latitude",
        "longitude",
        "gps_source",
        "detection_id",
        "class",
        "class_id",
        "confidence",
        "x1",
        "y1",
        "x2",
        "y2"
    ]

    writer = csv.DictWriter(
        output,
        fieldnames=fieldnames
    )

    writer.writeheader()

    for detection in detections:

        bbox = detection["bbox"]

        writer.writerow(
            {
                "image": filename,
                "timestamp": timestamp,
                "latitude": latitude,
                "longitude": longitude,
                "gps_source": gps_source,
                "detection_id": detection["detection_id"],
                "class": detection["class"],
                "class_id": detection["class_id"],
                "confidence": detection["confidence"],
                "x1": bbox["x1"],
                "y1": bbox["y1"],
                "x2": bbox["x2"],
                "y2": bbox["y2"]
            }
        )

    return output.getvalue().encode("utf-8")


# ============================================================
# SIDEBAR
# ============================================================

st.sidebar.title("⚙️ Analysis Settings")

confidence_threshold = st.sidebar.slider(
    "Confidence threshold",
    min_value=0.10,
    max_value=0.95,
    value=0.70,
    step=0.05
)

image_size = st.sidebar.selectbox(
    "Image size",
    [640, 768, 1024, 1280],
    index=2
)

st.sidebar.markdown("---")

st.sidebar.subheader("🧠 AI Model")

st.sidebar.write("YOLO11n-Seg")
st.sidebar.write("Shipwreck segmentation")
st.sidebar.write("Model: v2_1024")

st.sidebar.markdown("---")

st.sidebar.subheader("📍 GPS")

gps_mode = st.sidebar.radio(
    "GPS source",
    [
        "Manual GPS",
        "Image EXIF GPS"
    ]
)

manual_latitude = 0.0
manual_longitude = 0.0

if gps_mode == "Manual GPS":

    manual_latitude = st.sidebar.number_input(
        "Latitude",
        min_value=-90.0,
        max_value=90.0,
        value=0.0,
        step=0.000001,
        format="%.7f"
    )

    manual_longitude = st.sidebar.number_input(
        "Longitude",
        min_value=-180.0,
        max_value=180.0,
        value=0.0,
        step=0.000001,
        format="%.7f"
    )

st.sidebar.markdown("---")

st.sidebar.info(
    "Aquadex AI is a research prototype. "
    "Detected regions require human verification."
)


# ============================================================
# MAIN HEADER
# ============================================================

st.markdown(
    '<div class="main-title">🌊 Aquadex AI</div>',
    unsafe_allow_html=True
)

st.markdown(
    '<div class="subtitle">'
    'AI-assisted sonar shipwreck detection and segmentation'
    '</div>',
    unsafe_allow_html=True
)


# ============================================================
# LOAD MODEL
# ============================================================

try:

    model, model_path = load_model()

except Exception as e:

    st.error(
        f"Unable to load YOLO model: {e}"
    )

    st.stop()


# ============================================================
# MODEL INFORMATION
# ============================================================

with st.expander(
    "🧠 Model Information",
    expanded=False
):

    st.write(
        "**Model:** YOLO11n-Seg"
    )

    st.write(
        "**Task:** Detection + Segmentation"
    )

    st.write(
        "**Version:** sss_shipwreck_v2_1024"
    )

    st.write(
        "**Image size:**",
        image_size
    )

    st.write(
        "**Device:**",
        "CUDA" if model.device.type == "cuda" else "CPU"
    )

    st.write(
        "**Model file:**",
        model_path
    )


# ============================================================
# IMAGE UPLOAD
# ============================================================

st.header("📷 Input Sonar Image")

uploaded_file = st.file_uploader(
    "Upload a sonar image",
    type=[
        "png",
        "jpg",
        "jpeg",
        "tif",
        "tiff"
    ]
)


if uploaded_file is None:

    st.info(
        "Upload a sonar image to begin analysis."
    )

    st.stop()


# ============================================================
# LOAD IMAGE
# ============================================================

try:

    image = Image.open(
        uploaded_file
    ).convert("RGB")

except Exception as e:

    st.error(
        f"Unable to open image: {e}"
    )

    st.stop()


# ============================================================
# GPS PROCESSING
# ============================================================

exif_latitude = None
exif_longitude = None

try:

    original_image = Image.open(
        uploaded_file
    )

    exif_latitude, exif_longitude = (
        extract_gps_from_image(
            original_image
        )
    )

except Exception:

    exif_latitude = None
    exif_longitude = None


if gps_mode == "Image EXIF GPS":

    if (
        exif_latitude is not None
        and exif_longitude is not None
    ):

        final_latitude = exif_latitude
        final_longitude = exif_longitude
        final_gps_source = "Image EXIF"

        st.success(
            "📍 GPS coordinates detected from image EXIF metadata."
        )

    else:

        st.warning(
            "No GPS metadata was found in this image. "
            "Enter Manual GPS coordinates in the sidebar."
        )

        final_latitude = manual_latitude
        final_longitude = manual_longitude
        final_gps_source = "Manual fallback"

else:

    final_latitude = manual_latitude
    final_longitude = manual_longitude
    final_gps_source = "Manual Entry"


# ============================================================
# GPS DISPLAY
# ============================================================

st.markdown(
    "### 📍 Detection Location"
)

gps_col1, gps_col2, gps_col3 = st.columns(3)

with gps_col1:

    st.metric(
        "Latitude",
        f"{final_latitude:.7f}"
    )

with gps_col2:

    st.metric(
        "Longitude",
        f"{final_longitude:.7f}"
    )

with gps_col3:

    st.metric(
        "GPS Source",
        final_gps_source
    )


# ============================================================
# INPUT IMAGE
# ============================================================

col1, col2 = st.columns(2)

with col1:

    st.subheader("📷 Input Sonar Image")

    st.image(
        image,
        use_container_width=True
    )


# ============================================================
# ANALYZE BUTTON
# ============================================================

with col2:

    st.subheader("🧠 Aquadex AI Detection")

    analyze = st.button(
        "🌊 ANALYZE WITH AQUADEX AI",
        type="primary",
        use_container_width=True
    )


# ============================================================
# RUN MODEL
# ============================================================

if analyze:

    with st.spinner(
        "Running YOLO11n-Seg analysis..."
    ):

        try:

            results = model.predict(
                source=image,
                imgsz=image_size,
                conf=confidence_threshold,
                verbose=False,
                device="cpu"
            )

        except Exception as e:

            st.error(
                f"Model inference failed: {e}"
            )

            st.stop()


    # ========================================================
    # EXTRACT DETECTIONS
    # ========================================================

    detections = create_detection_rows(
        results,
        confidence_threshold
    )


    # ========================================================
    # ANNOTATED IMAGE
    # ========================================================

    annotated_image = annotate_image(
        image,
        results,
        confidence_threshold
    )

    with col2:

        st.image(
            annotated_image,
            caption="Detected shipwreck candidates",
            use_container_width=True
        )


    # ========================================================
    # ANALYSIS VALUES
    # ========================================================

    detection_count = len(
        detections
    )

    if detection_count > 0:

        highest_confidence = max(
            d["confidence"]
            for d in detections
        )

    else:

        highest_confidence = 0.0


    # ========================================================
    # STATUS
    # ========================================================

    if detection_count > 0:

        st.markdown(
            f"""
            <div class="status-box detected">
                🟢 SHIPWRECK CANDIDATE DETECTED
            </div>
            """,
            unsafe_allow_html=True
        )

    else:

        st.markdown(
            f"""
            <div class="status-box not-detected">
                🔴 NO CANDIDATE ABOVE THRESHOLD
            </div>
            """,
            unsafe_allow_html=True
        )


    # ========================================================
    # SUMMARY
    # ========================================================

    st.header("📊 Analysis Summary")

    m1, m2, m3, m4 = st.columns(4)

    with m1:

        st.metric(
            "Candidates",
            detection_count
        )

    with m2:

        st.metric(
            "Highest Confidence",
            f"{highest_confidence * 100:.1f}%"
        )

    with m3:

        st.metric(
            "Threshold",
            f"{confidence_threshold * 100:.0f}%"
        )

    with m4:

        st.metric(
            "GPS",
            "Available"
            if (
                final_latitude != 0
                or final_longitude != 0
            )
            else "Not specified"
        )


    # ========================================================
    # INDIVIDUAL DETECTIONS
    # ========================================================

    st.header("🔎 Individual Detections")

    if detection_count > 0:

        for detection in detections:

            bbox = detection["bbox"]

            with st.expander(
                f"Detection {detection['detection_id']} — "
                f"{detection['class']} — "
                f"{detection['confidence'] * 100:.1f}%"
            ):

                st.write(
                    "**Class:**",
                    detection["class"]
                )

                st.write(
                    "**Confidence:**",
                    f"{detection['confidence'] * 100:.2f}%"
                )

                st.write(
                    "**Bounding box:**",
                    bbox
                )

                st.write(
                    "**GPS:**",
                    f"{final_latitude:.7f}, "
                    f"{final_longitude:.7f}"
                )

    else:

        st.info(
            "No detection exceeded the selected confidence threshold."
        )


    # ========================================================
    # TIMESTAMP + FILENAMES
    # ========================================================

    timestamp = datetime.now().strftime(
        "%Y%m%d_%H%M%S"
    )

    original_name = os.path.splitext(
        uploaded_file.name
    )[0]

    base_name = (
        f"{original_name}_aquadex_{timestamp}"
    )


    image_filename = (
        base_name + ".png"
    )

    json_filename = (
        base_name + ".json"
    )

    csv_filename = (
        base_name + ".csv"
    )


    image_path = os.path.join(
        IMAGE_DIR,
        image_filename
    )

    json_path = os.path.join(
        JSON_DIR,
        json_filename
    )

    csv_path = os.path.join(
        CSV_DIR,
        csv_filename
    )


    # ========================================================
    # SAVE ANNOTATED IMAGE
    # ========================================================

    annotated_image.save(
        image_path
    )


    # ========================================================
    # JSON REPORT
    # ========================================================

    report = {

        "project": "Aquadex AI",

        "description":
            "AI-assisted sonar shipwreck detection and segmentation",

        "timestamp": timestamp,

        "image": {
            "filename": uploaded_file.name,
            "image_width": image.width,
            "image_height": image.height
        },

        "model": {
            "name": "YOLO11n-Seg",
            "version": "sss_shipwreck_v2_1024",
            "model_file": model_path,
            "task": "Detection + Segmentation",
            "image_size": image_size,
            "device": (
                "CUDA"
                if model.device.type == "cuda"
                else "CPU"
            )
        },

        "gps": {

            "latitude": final_latitude,

            "longitude": final_longitude,

            "source": final_gps_source

        },

        "analysis": {

            "confidence_threshold":
                confidence_threshold,

            "candidate_count":
                detection_count,

            "highest_confidence":
                highest_confidence

        },

        "detections":
            detections,

        "research_note":
            "AI-generated detections are potential "
            "shipwreck candidates and require human "
            "verification before being considered "
            "confirmed archaeological or maritime findings."

    }


    json_bytes = make_json_bytes(
        report
    )


    with open(
        json_path,
        "wb"
    ) as f:

        f.write(
            json_bytes
        )


    # ========================================================
    # CSV REPORT
    # ========================================================

    csv_bytes = make_csv_bytes(
        detections=detections,
        filename=uploaded_file.name,
        latitude=final_latitude,
        longitude=final_longitude,
        gps_source=final_gps_source,
        timestamp=timestamp
    )


    with open(
        csv_path,
        "wb"
    ) as f:

        f.write(
            csv_bytes
        )


    # ========================================================
    # EXPORT REPORTS
    # ========================================================

    st.header("📥 Export Reports")

    export1, export2, export3 = st.columns(3)

    with export1:

        st.download_button(
            label="📄 Download JSON",
            data=json_bytes,
            file_name=json_filename,
            mime="application/json",
            use_container_width=True
        )

    with export2:

        st.download_button(
            label="📊 Download CSV",
            data=csv_bytes,
            file_name=csv_filename,
            mime="text/csv",
            use_container_width=True
        )

    with export3:

        image_buffer = io.BytesIO()

        annotated_image.save(
            image_buffer,
            format="PNG"
        )

        st.download_button(
            label="🖼️ Download Image",
            data=image_buffer.getvalue(),
            file_name=image_filename,
            mime="image/png",
            use_container_width=True
        )


    # ========================================================
    # SAVED FILES
    # ========================================================

    st.header("💾 Saved to Google Drive")

    st.code(
        f"""
Annotated image:
{image_path}

JSON report:
{json_path}

CSV report:
{csv_path}
"""
    )


    # ========================================================
    # AI INTERPRETATION
    # ========================================================

    st.header("🧠 AI Interpretation")

    if detection_count > 0:

        st.write(
            f"Aquadex AI identified "
            f"**{detection_count} potential shipwreck "
            f"region(s)** above the selected "
            f"**{confidence_threshold * 100:.0f}% "
            f"confidence threshold**."
        )

        st.write(
            f"The highest-confidence candidate has "
            f"a confidence of "
            f"**{highest_confidence * 100:.1f}%**."
        )

    else:

        st.write(
            "Aquadex AI did not identify any candidate "
            "regions above the selected confidence threshold."
        )


    # ========================================================
    # GPS INTERPRETATION
    # ========================================================

    st.markdown(
        "### 📍 Location Information"
    )

    st.write(
        f"**Latitude:** {final_latitude:.7f}"
    )

    st.write(
        f"**Longitude:** {final_longitude:.7f}"
    )

    st.write(
        f"**GPS source:** {final_gps_source}"
    )


    # ========================================================
    # HUMAN VERIFICATION WARNING
    # ========================================================

    st.warning(
        "Research Prototype: AI-generated detections "
        "indicate potential shipwreck regions and should "
        "not be considered confirmed archaeological or "
        "maritime findings. All detections require "
        "human verification."
    )


# ============================================================
# FOOTER
# ============================================================

st.markdown("---")

st.caption(
    "🌊 Aquadex AI • AI-assisted sonar analysis "
    "for maritime exploration • Research Prototype"
)

st.caption(
    "Detection results require human verification."
)
'''

with open(APP_PATH, "w", encoding="utf-8") as f:
    f.write(app_code)

print("Aquadex AI application created:")
print(APP_PATH)

print("\nFile size:")
print(os.path.getsize(APP_PATH), "bytes")

Aquadex AI application created:
/content/aquadex_app.py

File size:
27752 bytes


In [46]:
# ============================================================
# CELL 5 — VERIFY APPLICATION
# ============================================================

import os

assert os.path.exists(
    "/content/aquadex_app.py"
)

print("✅ aquadex_app.py exists")

print("\nChecking model...")

assert os.path.exists(
    "/content/drive/MyDrive/Marine_Debris_Project/"
    "training/sss_shipwreck_v2_1024/weights/best.pt"
)

print("✅ YOLO v2 best.pt exists")

print("\nChecking output directories...")

for folder in [
    "/content/drive/MyDrive/Marine_Debris_Project/"
    "predictions/aquadex_ai/images",

    "/content/drive/MyDrive/Marine_Debris_Project/"
    "predictions/aquadex_ai/json",

    "/content/drive/MyDrive/Marine_Debris_Project/"
    "predictions/aquadex_ai/csv"
]:

    os.makedirs(folder, exist_ok=True)

    print("✅", folder)

print("\nEverything is ready.")

✅ aquadex_app.py exists

Checking model...
✅ YOLO v2 best.pt exists

Checking output directories...
✅ /content/drive/MyDrive/Marine_Debris_Project/predictions/aquadex_ai/images
✅ /content/drive/MyDrive/Marine_Debris_Project/predictions/aquadex_ai/json
✅ /content/drive/MyDrive/Marine_Debris_Project/predictions/aquadex_ai/csv

Everything is ready.


In [47]:
# ============================================================
# CELL 6 — START STREAMLIT
# ============================================================

import subprocess
import time
import os

# Stop old Streamlit processes
subprocess.run(
    "pkill -f 'streamlit run' || true",
    shell=True
)

time.sleep(2)

# Start Streamlit
log_file = open(
    "/content/streamlit.log",
    "w"
)

process = subprocess.Popen(
    [
        "streamlit",
        "run",
        "/content/aquadex_app.py",
        "--server.port",
        "8501",
        "--server.address",
        "0.0.0.0",
        "--server.headless",
        "true"
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT
)

time.sleep(5)

print("✅ Streamlit started")
print("PID:", process.pid)
print("Port: 8501")
print("\nOpening tunnel...")

✅ Streamlit started
PID: 15975
Port: 8501

Opening tunnel...


In [48]:
# ============================================================
# CELL 7 — CLOUDFLARE TUNNEL
# ============================================================

import os
import subprocess
import time
import re

# Check whether cloudflared exists
cloudflared_path = "/usr/local/bin/cloudflared"

if not os.path.exists(cloudflared_path):

    print("Installing cloudflared...")

    subprocess.run(
        [
            "wget",
            "-q",
            "-O",
            cloudflared_path,
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
        ],
        check=True
    )

    subprocess.run(
        ["chmod", "+x", cloudflared_path],
        check=True
    )

print("Starting Cloudflare tunnel...")

tunnel_log = open(
    "/content/cloudflared.log",
    "w"
)

tunnel_process = subprocess.Popen(
    [
        cloudflared_path,
        "tunnel",
        "--url",
        "http://127.0.0.1:8501"
    ],
    stdout=tunnel_log,
    stderr=subprocess.STDOUT
)

time.sleep(8)

print("Tunnel process started.")
print("\nCheck the URL below:")
print()

with open(
    "/content/cloudflared.log",
    "r"
) as f:

    log = f.read()

urls = re.findall(
    r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com",
    log
)

if urls:

    print("🌐 AQUADEX AI URL:")
    print(urls[-1])

else:

    print(
        "Tunnel URL not detected yet."
    )

    print(
        "Run this cell again after a few seconds."
    )

Starting Cloudflare tunnel...
Tunnel process started.

Check the URL below:

🌐 AQUADEX AI URL:
https://shareware-expires-meetup-intended.trycloudflare.com
